
# 0. More EDAs on Olympic data

In [0]:
# a)
import os
import pandas as pd

# Get notebook folder dynamically
notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
base_path = "/Workspace" + os.path.dirname(notebook_path)

# Read with pandas (uses local filesystem, no URI prefix needed)
pandas_df = pd.read_csv(f"{base_path}/data/athlete_events.csv")

# Convert to Spark dataframe
athlete_df = spark.createDataFrame(pandas_df)

In [0]:
# b)
athlete_df.columns

In [0]:
# c)
athlete_df.createOrReplaceTempView('athlete_df')

oldest_athletes = spark.sql("""
                            SELECT
                             Age,Sport
                            FROM athlete_df
                            ORDER BY Age DESC
                            LIMIT 10
                            """)

display(oldest_athletes)

In [0]:
# d)
athlete_df.createOrReplaceTempView('athlete_df')

youngest_athletes = spark.sql("""
                            SELECT
                             Age,Sport
                            FROM athlete_df
                            WHERE Age IS NOT NULL
                            ORDER BY Age ASC
                            LIMIT 10
                            """)

display(youngest_athletes)

In [0]:
# e)
athlete_df.createOrReplaceTempView('athlete_df')

highest_median_age = spark.sql("""
                            SELECT
                             Sport,
                             median(Age) as median_age
                            FROM athlete_df
                            GROUP BY Sport
                            ORDER BY median_age DESC
                            LIMIT 10
                            """)

display(highest_median_age)

In [0]:
# f)
athlete_df.createOrReplaceTempView('athlete_df')

lowest_median_age = spark.sql("""
                            SELECT
                             Sport,
                             median(Age) as median_age
                            FROM athlete_df
                            GROUP BY Sport
                            ORDER BY median_age ASC
                            LIMIT 10
                            """)

display(lowest_median_age)

In [0]:
# g)
athlete_df.createOrReplaceTempView('athlete_df')

most_gold_medals = spark.sql("""
                            SELECT
                             NOC,
                             count(Medal) as total_medals
                            FROM athlete_df
                            WHERE Medal = 'Gold'
                            GROUP BY NOC
                            ORDER BY total_medals DESC
                            LIMIT 10
                            """)

display(most_gold_medals)

In [0]:
# h)
athlete_df.createOrReplaceTempView('athlete_df')

most_medals = spark.sql("""
                            SELECT
                             NOC,
                             count(Medal) as total_medals
                            FROM athlete_df
                            WHERE Medal IN ('Gold', 'Silver', 'Bronze')
                            GROUP BY NOC
                            ORDER BY total_medals DESC
                            LIMIT 10
                            """)

display(most_medals)

In [0]:
# i)
import matplotlib.pyplot as plt
athlete_df.createOrReplaceTempView('athlete_df')

result = spark.sql("""
    SELECT
        Year,
        Sex,
        COUNT(*) as count
    FROM athlete_df
    WHERE Year IS NOT NULL
        AND Sex IS NOT NULL
    GROUP BY Year, Sex
    ORDER BY Year ASC
""").toPandas()

result.pivot(index='Year', columns='Sex', values='count').plot(
    kind='line',  
    figsize=(14, 6),
    title='Athletes by Year and Gender'
)

plt.xlabel('Year')
plt.ylabel('Count')
plt.tight_layout()
plt.show()